In [ ]:
import os

os.environ["TF_USE_LEGACY_KERAS"] = "1"
from tensorflow.keras.callbacks import EarlyStopping
import re
import time
import dotenv
import mlflow
import string
import dagshub
import logging
import pandas as pd
import tensorflow as tf
import tensorflow_hub as hub
import tensorflow_text as text
from IPython.display import clear_output
from tensorflow.keras.models import Model
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.feature_extraction.text import TfidfVectorizer
from tensorflow.keras.layers import Dense, Dropout, Input
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score


In [ ]:
import logging
import warnings

logging.basicConfig(
    level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s"
)

logging.getLogger().setLevel(logging.INFO)
warnings.simplefilter("ignore", UserWarning)
warnings.filterwarnings("ignore")

In [ ]:
imdb_df = pd.read_csv("imdb.csv")
imdb_df.shape

In [ ]:
df = imdb_df.sample(n=45000, random_state=42)
df.drop_duplicates(inplace=True)
df.to_csv(path_or_buf="sample.csv", index=False)
df.shape

In [ ]:
def remove_html(text):
    return re.sub(r"<.*?>", " ", text)


def remove_urls(text):
    return re.sub(r"http\S+|www\S+", " ", text)


def remove_punctuations(text):
    return text.translate(str.maketrans("", "", string.punctuation))


In [ ]:
def preprocess_text(text):
    text = text.lower()
    text = remove_html(text=text)
    text = remove_urls(text=text)
    text = remove_punctuations(text=text)
    return text


In [ ]:
df["review"] = df["review"].astype(str).apply(preprocess_text)
df["sentiment"] = df["sentiment"].map({"negative": 0, "positive": 1})
df

In [ ]:
test_size = 0.2
max_iter = 1000

keras_max_features = 30000
max_len = 400
embedding_dim = 200
cnn_epochs = 12
cnn_batch_size = 32
tfidf_max_features = 50000


In [ ]:
def create_hub_model():
    inputs = Input(shape=[], dtype=tf.string)
    embedding = hub.KerasLayer(
        "https://tfhub.dev/google/universal-sentence-encoder/4", trainable=True
    )(inputs)
    x = Dense(256, activation="relu")(embedding)
    x = Dropout(0.5)(x)
    x = Dense(64, activation="relu")(x)
    x = Dropout(0.5)(x)
    outputs = Dense(1, activation="sigmoid")(x)

    model = Model(inputs=inputs, outputs=outputs)
    model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    return model

In [ ]:
dotenv.load_dotenv()

dagshub_uri = os.getenv("DAGSHUB_URI")
dagshub_repo = os.getenv("DAGSHUB_REPO")
dagshub_username = os.getenv("DAGSHUB_USERNAME")

mlflow.set_tracking_uri(dagshub_uri)
dagshub.init(repo_owner=dagshub_username, repo_name=dagshub_repo, mlflow=True)

mlflow.set_experiment("Sentiment Analysis Experiments")
clear_output()

In [ ]:
# Tokenizer and padding removed for TF Hub usage
X_train, X_test, y_train, y_test = train_test_split(
    df["review"].values, df["sentiment"].values, test_size=test_size, random_state=42
)


In [ ]:
logging.info("Starting MLFlow run for Sentiment Analysis...")
t_start = time.time()

with mlflow.start_run(run_name="TF_Hub_USE_Experiment"):
    try:
        logging.info("Training Model...")

        mlflow.log_params(
            {
                "model_type": "TF_Hub_USE",
                "epochs": cnn_epochs,
                "batch_size": cnn_batch_size,
            }
        )

        model_hub = create_hub_model()

        early_stopping = EarlyStopping(
            monitor="val_loss", patience=3, restore_best_weights=True
        )

        model_hub.fit(
            X_train,
            y_train,
            epochs=cnn_epochs,
            batch_size=cnn_batch_size,
            validation_data=(X_test, y_test),
            callbacks=[early_stopping],
            verbose=0,
        )

        y_prob_hub = model_hub.predict(X_test)
        y_hat_hub = (y_prob_hub > 0.5).astype("int32")

        acc = accuracy_score(y_test, y_hat_hub)
        print(f"Accuracy: {acc}")
        mlflow.log_metrics(
            {
                "accuracy": acc,
                "precision": precision_score(y_test, y_hat_hub),
                "recall": recall_score(y_test, y_hat_hub),
                "f1_score": f1_score(y_test, y_hat_hub),
            }
        )

        # mlflow.keras.log_model(model_ann, "model")

    except Exception as e:
        logging.error(f"Training Error: {e}", exc_info=True)
        mlflow.log_param("error", f"Training Error: {str(e)}")


t_end = time.time()
logging.info(f"Total Execution time: {t_end - t_start:.2f} sec")